# Stage-B SFT Dataset Audit — the emitted train/val records

Audits the **final ms-swift SFT dataset** that the training job actually consumes:

- `sft_stageB/train.jsonl` (~20.2k) and `sft_stageB/val_1k.jsonl` (~1k)
- Built from the **corrected** judge (`..._recoord.jsonl`): xy2d/depth re-scored with the
  Qwen 0–1000 relative-coordinate rescale + tolerance, so xy2d is now included.

It is **not** a re-run of `audit_stage_b_winners.ipynb` (which audits the upstream
judge/trace data and the gate *funnel*). This notebook audits the *output* records
and verifies the gate held **on the rows that shipped**.

**Checks**
1. Load & count the emitted records.
2. Schema + structural-tag validation (`messages`/`images`, `<grounding><think><answer>` balance/order).
3. **Re-verify the >70% gate** — reconstruct each record's `id`, re-stream the judge file, confirm every emitted point has `answer_correctness == 1` on **≥12 / 16** teacher samples (≥75%, i.e. strictly >70%).
4. Image-file existence on disk.
5. Token-length budget (informs `--max_length`).
6. Train/val leakage + exact duplicates.
7. Content distributions (grounding objects, think/answer/question lengths) by `template_type`.
8. Qualitative viewer (image + question + target).
9. Summary PASS/FAIL table + JSON export.


## 0. Config & imports

In [ ]:

import os, json, re, hashlib, random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")

DATA_ROOT = Path("/mnt/data4/shasta/amar.amarjyoti/research_data/vlm_cot_distill")
SFT_DIR   = DATA_ROOT / "sft_stageB"
TRAIN_JSONL = SFT_DIR / "train.jsonl"
VAL_JSONL   = SFT_DIR / "val_1k.jsonl"

QA_FILE    = DATA_ROOT / "_train_qa_for_cot.jsonl"
JUDGE_FILE = DATA_ROOT / "judge_1778033257_qwen32b_recoord.jsonl"   # corrected Stage-B judge: xy2d/depth re-scored with Qwen 0-1000 relative-coord rescale + tolerance (10GB)

# Tokenizer for the length budget. Both 8B variants share the qwen3_vl tokenizer;
# Instruct is fine for counting text tokens.
MODEL_PATH = "/mnt/data4/shasta/amar.amarjyoti/research_data/models/Qwen3-VL-8B-Instruct"

# Constants that must match the build + job script.
N_SAMPLES   = 16     # gate denominator
MIN_CORRECT = 12     # gate: keep iff n_correct >= 12  (12/16 = 75% > 70%)
IMAGE_MAX_TOKEN_NUM = 1024   # matches IMAGE_MAX_TOKEN_NUM in pretrain_model_14.sh
JOB_MAXLEN  = 4096   # current --max_length in the job script

OUT_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
OUT_DIR = Path("sft_dataset_audit_out"); OUT_DIR.mkdir(exist_ok=True)
print("figures ->", OUT_DIR.resolve())

## 1. Load the emitted SFT records

In [ ]:

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for ln in f:
            ln = ln.strip()
            if ln:
                rows.append(json.loads(ln))
    return rows

train = load_jsonl(TRAIN_JSONL)
val   = load_jsonl(VAL_JSONL)
for r in train: r["_split"] = "train"
for r in val:   r["_split"] = "val"
records = train + val
print(f"train: {len(train):,}   val: {len(val):,}   total: {len(records):,}")
# Sanity (not magic numbers): both splits non-empty, and the image-disjoint val is
# at its ~1000 target (overshoot is bounded by the largest single held-out image).
assert len(train) > 0 and len(val) > 0, (len(train), len(val))
assert 1000 <= len(val) <= 1100, len(val)

## 2. Schema & structural-tag validation

Every record must be exactly:
`{"messages":[system,user,assistant], "images":[abs_path]}` with **one** `<image>`
token in the user turn, and an assistant target carrying exactly one balanced,
correctly-ordered `<grounding>…</grounding>` → `<think>…</think>` → `<answer>…</answer>`.

In [ ]:

SYS_EXPECTED_PREFIX = "You are a spatial visual-reasoning assistant."
TAGS = ["<grounding>","</grounding>","<think>","</think>","<answer>","</answer>"]

def assistant_text(r): return r["messages"][2]["content"]
def user_text(r):      return r["messages"][1]["content"]

problems = defaultdict(list)
empty_grounding = 0
sys_prompts = Counter()

for i, r in enumerate(records):
    keyset = set(r.keys()) - {"_split"}
    if keyset != {"messages", "images"}:
        problems["bad_toplevel_keys"].append(i)
    msgs = r.get("messages", [])
    if [m.get("role") for m in msgs] != ["system","user","assistant"]:
        problems["bad_roles"].append(i); continue
    sys_prompts[msgs[0]["content"]] += 1
    if not msgs[0]["content"].startswith(SYS_EXPECTED_PREFIX):
        problems["unexpected_system"].append(i)

    u = user_text(r)
    if u.count("<image>") != 1:       problems["image_token_count"].append(i)
    if not u.startswith("<image>"):   problems["image_token_not_prefix"].append(i)

    imgs = r.get("images", [])
    if not (isinstance(imgs, list) and len(imgs) == 1 and isinstance(imgs[0], str)):
        problems["bad_images_field"].append(i)

    a = assistant_text(r)
    # exactly one of each tag
    if any(a.count(t) != 1 for t in TAGS):
        problems["tag_count"].append(i)
    # correct order
    pos = [a.find(t) for t in TAGS]
    if any(p < 0 for p in pos) or pos != sorted(pos):
        problems["tag_order"].append(i)
    # grounding emptiness
    g = a[a.find("<grounding>")+len("<grounding>"):a.find("</grounding>")]
    if g.strip() == "":
        empty_grounding += 1

print("distinct system prompts:", len(sys_prompts))
print("empty <grounding></grounding> records:", empty_grounding, "(expected ~2257)")
print()
if problems:
    for k, v in problems.items():
        print(f"  PROBLEM {k}: {len(v)} (e.g. idx {v[:5]})")
else:
    print("ALL STRUCTURAL CHECKS PASSED — every record is schema- and tag-valid.")

## 3. Re-verify the >70% gate on the shipped records

The emitted records carry only `messages`+`images` — the original `id` was dropped.
Reconstruct it via the **unique** `(image_path, question)` key from the QA file,
then re-stream the judge file to recompute, per id,
`n_correct = #{ j : scores[j].answer_correctness == 1 }` over the fixed 16 samples.

**Assertion:** every shipped record's id has `n_correct >= 12` (≥75% > 70%).

In [ ]:

# QA lookup: (image_path, prompt) -> id, and id -> metadata
qa_key2id = {}
id2meta = {}
with open(QA_FILE) as f:
    for ln in f:
        r = json.loads(ln)
        qa_key2id[(r["image_path"], r["prompt"])] = r["id"]
        id2meta[r["id"]] = {"template_type": r.get("template_type",""),
                            "task_family":  r.get("task_family","")}
print("QA keys:", len(qa_key2id))

def record_id(r):
    q = user_text(r)[len("<image>"):]           # strip the literal <image> prefix
    return qa_key2id.get((r["images"][0], q))

ids = [record_id(r) for r in records]
unmapped = sum(1 for x in ids if x is None)
print("emitted records mapped to an id:", len(ids) - unmapped, "/", len(ids),
      "(unmapped:", unmapped, ")")

In [ ]:

# Stream the 10GB judge file ONCE: id -> n_correct (and best_idx).
n_correct_by_id = {}
best_idx_by_id  = {}
gate_hist = Counter()         # over ALL judged ids
read = 0
with open(JUDGE_FILE) as f:
    for ln in f:
        ln = ln.strip()
        if not ln: continue
        try:
            j = json.loads(ln)
        except json.JSONDecodeError:
            continue
        read += 1
        scores = j.get("scores") or []
        nc = sum(1 for s in scores if s is not None and s.get("answer_correctness") == 1)
        n_correct_by_id[j["id"]] = nc
        best_idx_by_id[j["id"]]  = j.get("best_idx")
        gate_hist[nc] += 1
print(f"judge records read: {read:,}")
print("kept ids at gate (n_correct>=12):", sum(v for k,v in gate_hist.items() if k>=MIN_CORRECT))

In [ ]:

# Per-EMITTED-record n_correct, and the hard assertion.
emit_nc = []
below_gate = []
for i, (r, rid) in enumerate(zip(records, ids)):
    if rid is None:
        continue
    nc = n_correct_by_id.get(rid)
    emit_nc.append(nc)
    if nc is None or nc < MIN_CORRECT:
        below_gate.append((i, rid, nc))

emit_nc = [x for x in emit_nc if x is not None]
dist = Counter(emit_nc)
print("n_correct distribution of SHIPPED records (should be 12..16 only):")
for k in range(N_SAMPLES + 1):
    if dist.get(k, 0):
        pct = 100 * dist[k] / len(emit_nc)
        print(f"  {k:2d}/16  ({k/16:5.1%}):  {dist[k]:6,}  ({pct:4.1f}%)")
print()
print("min n_correct among shipped:", min(emit_nc), " -> ", f"{min(emit_nc)/16:.1%} pass rate")
print("mean n_correct among shipped:", f"{np.mean(emit_nc):.2f} / 16  ({np.mean(emit_nc)/16:.1%})")
print()
if below_gate:
    print(f"!!! GATE VIOLATION: {len(below_gate)} shipped records below 12/16, e.g. {below_gate[:5]}")
else:
    print("GATE RE-VERIFIED: every shipped record has >=12/16 answer-correct (>=75% > 70%). PASS.")
GATE_OK = (len(below_gate) == 0) and (min(emit_nc) >= MIN_CORRECT)

In [ ]:

# Visualise: shipped pass-rate distribution + the gate line on the full judged population.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ks = list(range(N_SAMPLES + 1))
ax[0].bar(ks, [dist.get(k,0) for k in ks], color="#2a9d8f")
ax[0].axvline(MIN_CORRECT-0.5, color="crimson", ls="--", label="gate (>=12/16)")
ax[0].set(title="Shipped records by #correct / 16", xlabel="n_correct", ylabel="records")
ax[0].legend()
ax[1].bar(ks, [gate_hist.get(k,0) for k in ks], color="#888")
ax[1].bar([k for k in ks if k>=MIN_CORRECT],
          [gate_hist.get(k,0) for k in ks if k>=MIN_CORRECT], color="#2a9d8f")
ax[1].axvline(MIN_CORRECT-0.5, color="crimson", ls="--")
ax[1].set(title="Full judged population (green=kept)", xlabel="n_correct", ylabel="ids")
plt.tight_layout(); plt.savefig(OUT_DIR/"gate_recheck.png", dpi=110); plt.show()

## 4. Image-file existence

In [ ]:

imgs = [r["images"][0] for r in records]
uniq = set(imgs)
missing = sorted(p for p in uniq if not os.path.exists(p))
print(f"unique images: {len(uniq):,}   total references: {len(imgs):,}")
print(f"missing on disk: {len(missing)}")
if missing:
    print("  e.g.:", missing[:5])
else:
    print("ALL referenced images exist on disk. PASS.")
IMAGES_OK = (len(missing) == 0)

## 5. Token-length budget (informs `--max_length`)

Sequence length per record ≈ **text tokens** (system + user-question + assistant
target, via the qwen3_vl tokenizer) **+ image tokens** (capped at
`IMAGE_MAX_TOKEN_NUM = 1024` in the job script) + small chat-template overhead.
The image term is an upper-bound estimate; text is exact. Use this to choose
`--max_length` and to see how many records would truncate.

In [ ]:

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

CHAT_OVERHEAD = 24   # rough role/template token overhead

def text_tokens(r):
    sys = r["messages"][0]["content"]
    usr = user_text(r)[len("<image>"):]      # exclude the <image> placeholder
    asst = assistant_text(r)
    n = 0
    for t in (sys, usr, asst):
        n += len(tok(t, add_special_tokens=False)["input_ids"])
    return n

txt = np.array([text_tokens(r) for r in records])
est_total = txt + IMAGE_MAX_TOKEN_NUM + CHAT_OVERHEAD

def qtab(a, name):
    qs = [0.5, 0.9, 0.95, 0.99, 1.0]
    print(name)
    for q in qs:
        print(f"  p{int(q*100):>3}: {np.quantile(a, q):8.0f}")
    print(f"  mean: {a.mean():8.0f}")

qtab(txt, "TEXT tokens (system+question+target):")
print()
qtab(est_total, f"EST TOTAL seq len (text + {IMAGE_MAX_TOKEN_NUM} image + {CHAT_OVERHEAD}):")
print()
for cap in [2048, 3072, 4096, 6144, 8192]:
    frac = (est_total > cap).mean()
    flag = "  <-- job --max_length" if cap == JOB_MAXLEN else ""
    print(f"  est total > {cap:5d}: {frac:6.2%} of records truncated{flag}")

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(txt, bins=60, color="#264653"); ax[0].set(title="Text tokens / record", xlabel="tokens")
ax[1].hist(est_total, bins=60, color="#e76f51")
ax[1].axvline(JOB_MAXLEN, color="k", ls="--", label=f"max_length={JOB_MAXLEN}")
ax[1].set(title=f"Est. total seq len (+{IMAGE_MAX_TOKEN_NUM} image)", xlabel="tokens"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT_DIR/"token_lengths.png", dpi=110); plt.show()

## 6. Train/val leakage & exact duplicates

In [ ]:

train_ids = {record_id(r) for r in train} - {None}
val_ids   = {record_id(r) for r in val}   - {None}
overlap = train_ids & val_ids
print(f"train ids: {len(train_ids):,}   val ids: {len(val_ids):,}")
print(f"train∩val id overlap: {len(overlap)}  ", "PASS" if not overlap else "!!! LEAKAGE")

# Image-level disjointness (split is image-grouped: no val image may appear in train).
tr_img = {r["images"][0] for r in train}
va_img = {r["images"][0] for r in val}
img_overlap = tr_img & va_img
print(f"train imgs: {len(tr_img):,}   val imgs: {len(va_img):,}")
print(f"train∩val IMAGE overlap: {len(img_overlap)}  ",
      "PASS (image-disjoint)" if not img_overlap else "!!! IMAGE LEAKAGE")
IMG_DISJOINT_OK = (len(img_overlap) == 0)

def rec_hash(r):
    blob = r["images"][0] + "\x00" + user_text(r) + "\x00" + assistant_text(r)
    return hashlib.md5(blob.encode("utf-8")).hexdigest()

hashes = Counter(rec_hash(r) for r in records)
dups = {h: c for h, c in hashes.items() if c > 1}
print(f"exact-duplicate records (image+question+target): {sum(c-1 for c in dups.values())} "
      f"across {len(dups)} groups")
LEAK_OK = (len(overlap) == 0)

## 7. Content distributions by `template_type`

This section describes **what the target answers actually contain**, broken down by
question template. For every record we measure four things from the assistant target
and the question:

| Measured quantity | What it is |
|---|---|
| **num_grounding_objects** | how many `<objN>...</objN>` points appear inside the `<grounding>` block |
| **think_block_word_count** | number of words inside the `<think>...</think>` reasoning block |
| **answer_block_word_count** | number of words inside the final `<answer>...</answer>` block |
| **question_word_count** | number of words in the user question (image token stripped) |

**Table 1 — per-template summary.** One row per `template_type`, sorted by how many
records use that template. Columns:

- **num_records** — how many SFT records use this template.
- **median_grounding_objects** — typical number of grounded objects per record.
- **median_think_words** — typical reasoning-block length (words).
- **median_answer_words** — typical final-answer length (words).

("median" = the middle value, so it's robust to a few very long outliers. We use it
instead of the mean for that reason.)

**Table 2 — overall distribution** (`describe`) gives count / mean / std / min /
quartiles / max for each of the four measured quantities across **all** records,
regardless of template.

In [ ]:

OBJ_RE = re.compile(r"<obj\d+>")
def n_objs(r):
    a = assistant_text(r)
    g = a[a.find("<grounding>"):a.find("</grounding>")]
    return len(OBJ_RE.findall(g))
def block(a, tag):
    return a[a.find(f"<{tag}>")+len(tag)+2 : a.find(f"</{tag}>")]

rows = []
for r, rid in zip(records, ids):
    a = assistant_text(r)
    meta = id2meta.get(rid, {})
    rows.append({
        "split": r["_split"],
        "template_type": meta.get("template_type",""),
        "task_family":   meta.get("task_family",""),
        "num_grounding_objects":   n_objs(r),
        "think_block_word_count":  len(block(a,"think").split()),
        "answer_block_word_count": len(block(a,"answer").split()),
        "question_word_count":     len(user_text(r)[len("<image>"):].split()),
    })
df = pd.DataFrame(rows)

print("=== Table 1: per-template_type summary (sorted by record count) ===")
summary = df.groupby("template_type").agg(
    num_records=("num_grounding_objects", "size"),
    median_grounding_objects=("num_grounding_objects", "median"),
    median_think_words=("think_block_word_count", "median"),
    median_answer_words=("answer_block_word_count", "median"),
).sort_values("num_records", ascending=False)
print(summary.to_string())

print("\n=== Table 2: overall distribution across ALL records ===")
print("(count / mean / std / min / 25% / 50%(median) / 75% / max)")
print(df[["num_grounding_objects", "think_block_word_count",
          "answer_block_word_count", "question_word_count"]].describe().round(1).to_string())

In [ ]:

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for axi, col, title in zip(ax.ravel(),
        ["num_grounding_objects","think_block_word_count","answer_block_word_count","question_word_count"],
        ["# grounding objects per record","think-block words per record",
         "answer-block words per record","question words per record"]):
    axi.hist(df[col], bins=40, color="#457b9d")
    axi.set(title=title, xlabel=title, ylabel="# records")
plt.tight_layout(); plt.savefig(OUT_DIR/"content_dists.png", dpi=110); plt.show()

# template mix bar
fig, axx = plt.subplots(figsize=(10,4))
df["template_type"].value_counts().plot.bar(ax=axx, color="#2a9d8f")
axx.set(title="records per template_type", ylabel="records")
plt.tight_layout(); plt.savefig(OUT_DIR/"template_mix.png", dpi=110); plt.show()

## 8. Qualitative viewer — image + question + target

In [ ]:

from PIL import Image
def show(r):
    rid = record_id(r)
    print("="*100)
    print("id:", rid, "| template:", id2meta.get(rid,{}).get("template_type",""),
          "| n_correct:", n_correct_by_id.get(rid))
    print("Q:", user_text(r)[len("<image>"):][:300])
    print("-"*100)
    print(assistant_text(r)[:900])
    try:
        im = Image.open(r["images"][0]).convert("RGB")
        plt.figure(figsize=(6,4)); plt.imshow(im); plt.axis("off"); plt.show()
    except Exception as e:
        print("(image load failed:", e, ")")

rng = random.Random(0)
for r in rng.sample(train, 3):
    show(r)

## 9. Summary — PASS/FAIL & JSON export

In [ ]:

summary = {
    "train": len(train), "val": len(val), "total": len(records),
    "structural_problems": {k: len(v) for k, v in problems.items()},
    "empty_grounding": int(empty_grounding),
    "gate": {
        "min_correct_threshold": MIN_CORRECT,
        "min_shipped_n_correct": int(min(emit_nc)),
        "mean_shipped_n_correct": float(np.mean(emit_nc)),
        "below_gate_count": len(below_gate),
        "unmapped_records": int(unmapped),
        "PASS": bool(GATE_OK),
    },
    "images_all_exist": bool(IMAGES_OK), "missing_images": len(missing),
    "train_val_id_leakage": len(overlap), "leakage_PASS": bool(LEAK_OK),
    "train_val_image_overlap": len(img_overlap), "image_disjoint_PASS": bool(IMG_DISJOINT_OK),
    "exact_duplicate_records": int(sum(c-1 for c in dups.values())),
    "maxlen_job": JOB_MAXLEN,
    "frac_truncated_at_job_maxlen": float((est_total > JOB_MAXLEN).mean()),
    "p99_est_total_tokens": float(np.quantile(est_total, 0.99)),
}
checks = {
    "structural valid":   not problems,
    "gate >70% verified": GATE_OK,
    "all images exist":   IMAGES_OK,
    "no train/val id leak": LEAK_OK,
    "image-disjoint val":   IMG_DISJOINT_OK,
}
print("===== AUDIT SUMMARY =====")
for k, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {k}")
print()
print(json.dumps(summary, indent=2))
with open(OUT_DIR/"audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nwrote", (OUT_DIR/"audit_summary.json").resolve())